In [1]:
import numpy as np
import h5py
import torch
from score_models import NCSNpp, ScoreModel
import torch.nn.functional as F
import os, re, json
from astropy.visualization import ImageNormalize, LogStretch, SqrtStretch
from glob import glob
from astropy.io import fits
from astropy import units
from functorch import vjp, vmap, grad
from astropy.wcs import WCS
from tqdm import tqdm
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord

# import sys
# caution: path[0] is reserved for script path (or '' in REPL)
# sys.path.insert(1, '../scripts')
# import forward_model


DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Construct forward model

In [2]:
def make_wcs(skycoord, orientation, pixels, pixel_size):  
    hdr = fits.Header()
    hdr["NAXIS1"] = pixels
    hdr["NAXIS2"] = pixels
    hdr["CRVAL1"] = skycoord.ra.to(units.deg).value
    hdr["CRVAL2"] = skycoord.dec.to(units.deg).value
    hdr["CRPIX1"] = pixels/2 - 0.5 # Same convention as Cutout2D
    hdr["CRPIX2"] = pixels/2 - 0.5
    hdr["CUNIT1"] = 'deg'
    hdr["CUNIT2"] = 'deg'
    hdr["CTYPE1"] = "RA---TAN"
    hdr["CTYPE2"] = "DEC--TAN"
    hdr["CDELT1"] = pixel_size.to(units.deg).value
    hdr["CDELT2"] = pixel_size.to(units.deg).value
    theta = orientation * np.pi / 180
    rotation = np.array([[np.cos(theta), -np.sin(theta)], 
                         [np.sin(theta), np.cos(theta)]])
    mirror_j = np.array([[1, 0], [0, -1]])
    pc = rotation @ mirror_j
    hdr["PC1_1"] = pc[0, 0]
    hdr["PC1_2"] = pc[0, 1]
    hdr["PC2_1"] = pc[1, 0]
    hdr["PC2_2"] = pc[1, 1]
    return WCS(hdr)
    
def make_forward_model(
        psf:np.ndarray, 
        wcs_list:list[WCS, ...], 
        super_sampling_factor:int,
        model_pixels:int,
        model_pixel_size:units.Quantity,
        zero_padding:int=0,
        fiducial_center:SkyCoord=None,
        fiducial_orientation:float=None, # Pick the orientation of the first WCS, angle East of North
        **kwargs
        ):
    if psf.ndim == 2:
        C = 1
        H, W = psf.shape
    elif psf.ndim == 3:
        C, H, W = psf.shape
    psf = torch.tensor(psf).float().to(DEVICE).view(C, 1, H, W) # reshape to a convolution kernel [channel_out, channels_in/groups, H, W]
    batched_interpolation = vmap(interpolate, in_dims=(0, None))  # only batch over the images (first argument of interpolate)
    
    if fiducial_center is None:
        # Use same convention as Cutout2D for reference pixel
        center = [dim / 2 - 0.5 for dim in wcs_list[0].pixel_shape]
        fiducial_center = wcs_list[0].pixel_to_world(*center)
    if fiducial_orientation is None:
        pc = wcs_list[0].pixel_scale_matrix
        fiducial_orientation = np.arctan2(pc[1, 0], pc[0, 0]) * 180 / np.pi
    fiducial_wcs = make_wcs(fiducial_center, fiducial_orientation, model_pixels, model_pixel_size)

    # Prepare coordinate systems
    model_coordinates_list = []
    for wcs in wcs_list:
        # Observation pixel coordinates super sampled
        u = (np.arange(super_sampling_factor * wcs.pixel_shape[0]) + 0.5) / super_sampling_factor 
        v = (np.arange(super_sampling_factor * wcs.pixel_shape[1]) + 0.5) / super_sampling_factor 
        # v = np.flip(v) # Remember matrix convention for pixel indexing
        u, v = np.meshgrid(u, v, indexing="ij")
        world = wcs.pixel_to_world(u, v)
        model_coordinates = np.stack(fiducial_wcs.world_to_pixel(world), axis=0)
        model_coordinates_list.append(torch.tensor(model_coordinates).float().to(DEVICE))
    
    def A(x):
        x = F.pad(x, pad=[zero_padding]*4, mode="constant", value=0.)
        ys = []
        for i in range(len(wcs_list)):
            y = batched_interpolation(x, model_coordinates_list[i])
            y = F.conv2d(y, psf, groups=C, padding="same")
            y = F.avg_pool2d(y, kernel_size=super_sampling_factor, stride=super_sampling_factor)
            # Add PSF model
            
            ys.append(y)
        return torch.concat(ys, dim=1)
    return A

# Load model

In [12]:
prior_path = "../models/ncsnpp_ct_g_220912024942"
prior_model = ScoreModel(checkpoints_directory=prior_path)

slic_path = "../models/ncsnpp_hst_noise_psf_f814w_wfc3uv_230603210517"
slic_model = ScoreModel(checkpoints_directory=slic_path)

In [13]:
# In principle, SLIC is trained with the same noise schedule, so we can define a common sigma_mn and sigma_max
sigma_min = prior_model.sde.sigma_min
sigma_max = prior_model.sde.sigma_max

def sigma(t): # scale of the marginal prob. distiribution
    return sigma_min * (sigma_max / sigma_min)**t.view(-1, 1, 1, 1)
def g(t): # diffusion coefficient of the VESDE
    return sigma(t) * np.sqrt(2 * (np.log(sigma_max) - np.log(sigma_min)))

# Load data

In [18]:
# Pick target and its coordinate
i = 0

target_names = {
    0: "J030000",
    1: "J085053",
    2: "J102036",
    3: "J105102",
    4: "J112526"
}
J030000 = SkyCoord('03h00m00.573s', '+00d48m28.01s', frame='icrs')
J085053 = SkyCoord('08h50m53.125s', '+44d51m22.50s', frame='icrs')
J102036 = SkyCoord('10h20m36.103s', '+60d23m38.96s', frame='icrs')
J105102 = SkyCoord('10h51m02.775s', '+52d50m49.86s', frame='icrs')
J112526 = SkyCoord('11h25m26.126s', '+00d29m01.32s', frame='icrs')
coordinates = {
    0: J030000,
    1: J085053,
    2: J102036,
    3: J105102,
    4: J112526
}
paths = {
    0: "../data/joseph_target_J030000.fits",
    1: "../data/joseph_target_J085053.fits",
    2: "../data/joseph_target_J102036.fits",
    3: "../data/joseph_target_J105102.fits",
    4: "../data/joseph_target_J112526.fits"
}

data = fits.open(paths[i])
coord = coordinates[i]
name = target_names[i]
print(name)
data.info()

J030000
Filename: ../data/joseph_target_J030000.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     909   ()      
  1  DRZ           1 ImageHDU        30   (64, 64)   float32   
  2  FLC           1 ImageHDU        31   (64, 64)   float64   
  3  FLC           2 ImageHDU        31   (64, 64)   float64   
  4  FLC           3 ImageHDU        31   (64, 64)   float64   
  5  FLC           4 ImageHDU        31   (64, 64)   float64   
  6  FLC           5 ImageHDU        31   (64, 64)   float64   
  7  FLC           6 ImageHDU        31   (64, 64)   float64   
  8  FLC           7 ImageHDU        31   (64, 64)   float64   
  9  FLC           8 ImageHDU        31   (64, 64)   float64   
 10  FLC           9 ImageHDU        31   (64, 64)   float64   
